## TCGA COAD (cBioPortal) — Hippo-stratified DEGs Analysis

Because this cBioPortal cohort may have **no adjacent normals**, we will:
- use **Tumor-only** samples
- compute Hippo/YAP scores
- compare **Hippo-high vs Hippo-low** tumors (top 25% vs bottom 25%)

Outputs:
- `data_processed/tcga/tcga_log2_rsem.csv`
- `results/hippo/tcga_hippo_yap_scores.csv`
- `results/de/tcga/TCGA_DEG_HippoHigh_vs_HippoLow.csv`
- volcano plot in `results/figures/tcga/`


In [ ]:
# --- Make sure expr is genes x samples and numeric ---
expr = expr.dropna(subset=[gene_col]).copy()
expr[gene_col] = expr[gene_col].astype(str).str.strip()
expr = expr.set_index(gene_col)

# drop entrez if present
for drop_c in ["Entrez_Gene_Id", "entrez_gene_id", "Entrez Gene Id"]:
    if drop_c in expr.columns:
        expr = expr.drop(columns=[drop_c])

expr = expr.apply(pd.to_numeric, errors="coerce").fillna(0)

# merge duplicate genes
if expr.index.duplicated().sum() > 0:
    expr = expr.groupby(expr.index).mean()

print("Expression matrix:", expr.shape)

# --- clinical: detect sample id + type columns (your code likely already did) ---
import re
cand_id = [c for c in clin.columns if re.search("sample|barcode|submitter", c, flags=re.I)]
sample_id_col = cand_id[0] if cand_id else clin.columns[0]
clin[sample_id_col] = clin[sample_id_col].astype(str).str.strip()

cand_type = [c for c in clin.columns if re.search("sample_type|tumor|normal", c, flags=re.I)]
type_col = cand_type[0] if cand_type else None

if type_col:
    clin[type_col] = clin[type_col].astype(str).str.lower()
    clin["group"] = np.where(clin[type_col].str.contains("normal"), "Normal", "Tumor")
else:
    clin["group"] = "Tumor"   # fallback

print("Clinical group counts:")
display(clin["group"].value_counts())

# --- overlap ---
expr_cols = pd.Index(expr.columns.astype(str).str.strip())
clin_ids  = pd.Index(clin[sample_id_col].astype(str).str.strip())
overlap = expr_cols.intersection(clin_ids)

print("Overlap samples:", len(overlap))
expr = expr.loc[:, overlap]
clin = clin[clin[sample_id_col].isin(overlap)].copy()
clin = clin.set_index(sample_id_col).loc[overlap].reset_index()

print("Aligned expr:", expr.shape)
print("Aligned clin:", clin.shape)

# Save clinical clean
clin.to_csv(f"{OUT_PROC}/tcga_clinical_sample_clean.csv", index=False)
print("Saved ->", f"{OUT_PROC}/tcga_clinical_sample_clean.csv")


In [ ]:
# Check normals
n_norm = int((clin["group"] == "Normal").sum())
n_tum  = int((clin["group"] == "Tumor").sum())

print(f"Tumor={n_tum} | Normal={n_norm}")

# If no/too few normals, proceed with tumor-only analysis
tumor_ids = clin.loc[clin["group"] == "Tumor", sample_id_col].astype(str).tolist()

expr_tumor = expr.loc[:, tumor_ids]
clin_tumor = clin.loc[clin["group"] == "Tumor"].copy()

print("Tumor-only expr:", expr_tumor.shape)
print("Tumor-only clin:", clin_tumor.shape)


In [ ]:
expr_log = np.log2(expr_tumor + 1)
expr_log.to_csv(f"{OUT_PROC}/tcga_log2_rsem.csv")
print("Saved ->", f"{OUT_PROC}/tcga_log2_rsem.csv")


**Hippo/YAP gene sets + scoring**

In [ ]:
def zscore_rows(expr_df):
    mu = expr_df.mean(axis=1)
    sd = expr_df.std(axis=1).replace(0, np.nan)
    return expr_df.sub(mu, axis=0).div(sd, axis=0)

def geneset_score(expr_log_df, genes):
    genes_present = [g for g in genes if g in expr_log_df.index]
    if len(genes_present) < 2:
        raise ValueError(f"Too few genes found from set. Found: {genes_present}")
    Z = zscore_rows(expr_log_df.loc[genes_present]).fillna(0)
    return Z.mean(axis=0)

HIPPO_CORE = ["STK3","STK4","SAV1","MOB1A","MOB1B","LATS1","LATS2","NF2"]
YAP_TAZ    = ["YAP1","WWTR1","TEAD1","TEAD2","TEAD3","TEAD4"]
YAP_TARGETS= ["CTGF","CYR61","ANKRD1","AREG","AMOTL2","BIRC5","FSTL1"]

scores = pd.DataFrame(index=expr_log.columns)
scores["hippo_core"]  = geneset_score(expr_log, HIPPO_CORE)
scores["yap_taz"]     = geneset_score(expr_log, YAP_TAZ)
scores["yap_targets"] = geneset_score(expr_log, YAP_TARGETS)

scores.to_csv(f"{BASE}/results/hippo/tcga_hippo_yap_scores.csv")
print("Saved ->", f"{BASE}/results/hippo/tcga_hippo_yap_scores.csv")
scores.head()


Stratify Hippo-high vs Hippo-low (top/bottom 25%)

In [ ]:
score_col = "yap_targets"   # you can switch to hippo_core if you prefer

q_low  = scores[score_col].quantile(0.25)
q_high = scores[score_col].quantile(0.75)

scores["hippo_group"] = "mid"
scores.loc[scores[score_col] <= q_low,  "hippo_group"] = "Hippo_low"
scores.loc[scores[score_col] >= q_high, "hippo_group"] = "Hippo_high"

print(scores["hippo_group"].value_counts())

keep = scores["hippo_group"].isin(["Hippo_high","Hippo_low"])
expr_sub = expr_log.loc[:, keep]
grp = scores.loc[keep, "hippo_group"].values

print("Expr for DEG:", expr_sub.shape)
print("Hippo-high:", (grp=="Hippo_high").sum(), "Hippo-low:", (grp=="Hippo_low").sum())


Safe Welch DEG (prevents NaN FDR)

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

def deg_welch_safe(expr_log_df, groups, g1="Hippo_high", g2="Hippo_low", min_var=1e-8):
    groups = np.array(groups, dtype=str)
    i1 = np.where(groups == g1)[0]
    i2 = np.where(groups == g2)[0]
    print(f"Group sizes: {g1}={len(i1)} | {g2}={len(i2)}")

    X1 = expr_log_df.iloc[:, i1]
    X2 = expr_log_df.iloc[:, i2]

    mean1 = X1.mean(axis=1)
    mean2 = X2.mean(axis=1)
    logFC = mean1 - mean2

    var1 = X1.var(axis=1)
    var2 = X2.var(axis=1)
    valid_var = (var1 > min_var) | (var2 > min_var)

    pvals = np.full(expr_log_df.shape[0], np.nan, dtype=float)

    idx = np.where(valid_var.values)[0]
    for i in idx:
        pvals[i] = ttest_ind(X1.iloc[i,:], X2.iloc[i,:], equal_var=False, nan_policy="omit").pvalue

    padj = np.ones_like(pvals, dtype=float)
    valid_p = np.isfinite(pvals)
    if valid_p.sum() > 0:
        padj[valid_p] = multipletests(pvals[valid_p], method="fdr_bh")[1]

    out = pd.DataFrame({
        "gene": expr_log_df.index.astype(str),
        f"mean_{g1}": mean1.values,
        f"mean_{g2}": mean2.values,
        f"logFC_{g1}_minus_{g2}": logFC.values,
        "pval": pvals,
        "padj_fdr": padj
    }).sort_values(["padj_fdr","pval"], ascending=[True, True])

    return out

deg_tcga = deg_welch_safe(expr_sub, grp, "Hippo_high", "Hippo_low")
display(deg_tcga.head(15))

deg_tcga.to_csv(f"{OUT_DE}/TCGA_DEG_HippoHigh_vs_HippoLow.csv", index=False)
print("Saved ->", f"{OUT_DE}/TCGA_DEG_HippoHigh_vs_HippoLow.csv")


Hippo HIgh vs. Hippo Low (Volcano plot)


In [ ]:
df = deg_tcga.copy()
df["neglog10_fdr"] = -np.log10(df["padj_fdr"] + 1e-300)

plt.figure(figsize=(7,5))
plt.scatter(df["logFC_Hippo_high_minus_Hippo_low"], df["neglog10_fdr"], alpha=0.35)

plt.axvline(1, linestyle="--")
plt.axvline(-1, linestyle="--")
plt.axhline(-np.log10(0.05), linestyle="--")

plt.title("TCGA COAD Tumors: Hippo-high vs Hippo-low")
plt.xlabel("log2FC (Hippo-high - Hippo-low)")
plt.ylabel("-log10(FDR)")
plt.tight_layout()

out_png = f"{OUT_FIG}/volcano_TCGA_HippoHigh_vs_HippoLow.png"
plt.savefig(out_png, dpi=200)
plt.show()

print("Saved ->", out_png)


Top 10 labeled DEGs

In [ ]:
top = df.sort_values("padj_fdr").head(10)

plt.figure(figsize=(7,5))
plt.scatter(df["logFC_Hippo_high_minus_Hippo_low"], df["neglog10_fdr"], alpha=0.20)
plt.axvline(1, linestyle="--"); plt.axvline(-1, linestyle="--")
plt.axhline(-np.log10(0.05), linestyle="--")

for _, r in top.iterrows():
    plt.text(r["logFC_Hippo_high_minus_Hippo_low"], r["neglog10_fdr"], r["gene"], fontsize=8)

plt.title("TCGA: Hippo-high vs Hippo-low (Top 10 labeled)")
plt.xlabel("log2FC"); plt.ylabel("-log10(FDR)")
plt.tight_layout()
plt.show()
